In [2]:
import re
from collections import defaultdict

In [10]:
# First Script: Merge all same-type pops ignoring culture and religion. 
# Culture% then gets equalized by proportionally increasing/decreasing the farmer pop sizes


# CONFIGURATION
PRIMARY_CULTURE = None  # auto-pick largest culture per province
PRIMARY_TYPES = [
    "aristocrats", "artisans", "bureaucrats", "capitalists",
    "clergymen", "clerks", "craftsmen", "farmers",
    "labourers", "officers", "slaves", "soldiers"
]
SECONDARY_TYPES = ["farmers"] # Merged Pops get converted to that

ARTISAN_CHUNK_SIZE = 20000
ARTISAN_MIN_CHUNK = 5000 # optional

# PARSER
def parse_pops(block):
    pops = []
    
    pattern = re.compile(
        r'(\w+)\s*=\s*{[^}]*?culture\s*=\s*(\w+)[^}]*?religion\s*=\s*(\w+)[^}]*?size\s*=\s*(\d+)[^}]*?}',
        re.DOTALL
    )

    for match in pattern.finditer(block):
        pop_type, culture, religion, size = match.groups()
        pops.append({
            "type": pop_type,
            "culture": culture,
            "religion": religion,
            "size": int(size)
        })

    return pops


# CORE TRANSFORMATION
def process_province(pops):
    # Step 1: aggregate by (type, culture, religion)
    triple_sum = defaultdict(int)
    for p in pops:
        ptype = normalize_type(p["type"])
        key = (ptype, p["culture"], p["religion"])
        triple_sum[key] += p["size"]

    # Step 2: collapse religion → (type, culture)
    collapsed = {}
    religion_tracker = defaultdict(lambda: defaultdict(int))

    for (ptype, culture, religion), size in triple_sum.items():
        collapsed.setdefault((ptype, culture), 0)
        collapsed[(ptype, culture)] += size
        religion_tracker[(ptype, culture)][religion] += size

    collapsed_data = {}
    for key, total_size in collapsed.items():
        religion_counts = religion_tracker[key]
        dominant_religion = max(religion_counts, key=religion_counts.get)
        collapsed_data[key] = {
            "size": total_size,
            "religion": dominant_religion
        }

    # Step 3: determine primary culture
    culture_totals = defaultdict(int)
    for (ptype, culture), data in collapsed_data.items():
        culture_totals[culture] += data["size"]

    if PRIMARY_CULTURE:
        primary = PRIMARY_CULTURE
    else:
        primary = max(culture_totals, key=culture_totals.get)
    # find dominant religion for this culture
    culture_religions = defaultdict(int)

    for (ptype, c), data in collapsed_data.items():
        if c == primary:
            culture_religions[data["religion"]] += data["size"]

    dominant = max(culture_religions, key=culture_religions.get)
    
    # Step 4: compute total per type
    type_totals = defaultdict(int)
    for (ptype, culture), data in collapsed_data.items():
        type_totals[ptype] += data["size"]

    # Step 5: compute total per culture
    culture_totals = defaultdict(int)
    for (ptype, culture), data in collapsed_data.items():
        culture_totals[culture] += data["size"]

    final = []

    # PRIMARY CULTURE
    for ptype, total_size in type_totals.items():
    
        if ptype == "farmers":
            continue  # handled later
    
        if ptype == "artisans":
            chunks = split_artisans(
                total_size,
                ARTISAN_CHUNK_SIZE,
                ARTISAN_MIN_CHUNK
            )
            for chunk_size in chunks:
                final.append({
                    "type": "artisans",
                    "culture": primary,
                    "religion": collapsed_data.get((ptype, primary), {}).get("religion", "unknown"),
                    "size": chunk_size
                })
        else:
            final.append({
                "type": ptype,
                "culture": primary,
                "religion": collapsed_data.get((ptype, primary), {}).get("religion", "unknown"),
                "size": total_size
            })

    #NON-PRIMARY CULTURES (farmers only)
    non_primary_farmers_total = 0

    for culture, total_size in culture_totals.items():
        if culture == primary:
            continue

        final.append({
            "type": "farmers",
            "culture": culture,
            "religion": collapsed_data.get(("farmers", culture), {}).get("religion", "unknown"),
            "size": total_size
        })

        non_primary_farmers_total += total_size

    # PRIMARY FARMERS
    total_population = sum(culture_totals.values())

    primary_other_types = sum(
        size for ptype, size in type_totals.items() if ptype != "farmers"
    )

    primary_farmers = total_population - primary_other_types - non_primary_farmers_total

    final.append({
        "type": "farmers",
        "culture": primary,
        "religion": collapsed_data.get(("farmers", primary), {}).get("religion", "unknown"),
        "size": primary_farmers
    })
    
    return final

#HELPERS
# Farmer/Labourer Helper
def normalize_type(ptype):
    if ptype in ["farmers", "labourers"]:
        return "farmers"
    return ptype


# Artisan Splitter / Helper
def split_artisans(size, chunk_size, min_chunk):
    chunks = []

    while size > chunk_size:
        chunks.append(chunk_size)
        size -= chunk_size

    # handle remainder
    if size > 0:
        if size < min_chunk and chunks:
            # merge small remainder into last chunk
            chunks[-1] += size
        else:
            chunks.append(size)

    return chunks



# WRITER
def format_province(province_id, pops, indent="\t"):
    lines = [f"{province_id} = {{"]

    for p in pops:
        lines.append(f"{indent}{p['type']} = {{")
        lines.append(f"{indent}\tculture = {p['culture']}")
        lines.append(f"{indent}\treligion = {p['religion']}")
        lines.append(f"{indent}\tsize = {p['size']}")
        lines.append(f"{indent}}}")
        lines.append("")

    lines.append("}")
    return "\n".join(lines)


# MAIN FILE PROCESSOR
def process_file(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Matches regex of full province blocks (including {} )
    province_pattern = re.compile(
        r'(\d+)\s*=\s*{([^{}]*(?:{[^{}]*}[^{}]*)*)}',
        re.DOTALL
    )

    def replace_province(match):
        province_id = match.group(1)
        original_block = match.group(0)
        inner_content = match.group(2)

        pops = parse_pops(inner_content)

        # if no pops found, do not execute
        if not pops:
            return original_block

        new_pops = process_province(pops)

        # Preserve original indentation
        indent_match = re.search(r'\n(\s*)\w+\s*=', inner_content)
        indent = indent_match.group(1) if indent_match else "\t"

        # Rebuilds the inside with extracted type, culture, religion & new size
        new_inner = []
        for p in new_pops:
            new_inner.append(f"{indent}{p['type']} = {{")
            new_inner.append(f"{indent}\tculture = {p['culture']}")
            new_inner.append(f"{indent}\treligion = {p['religion']}")
            new_inner.append(f"{indent}\tsize = {p['size']}")
            new_inner.append(f"{indent}}}")
            new_inner.append("")

        new_inner_str = "\n".join(new_inner).rstrip()

        return f"{province_id} = {{\n{new_inner_str}\n}}"

    # Replace province blocks in-place
    new_content = province_pattern.sub(replace_province, content)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(new_content)


# RUN
if __name__ == "__main__":
    process_file("China.txt", "China - simplified.txt")

In [8]:
# Second One: Simply merges all same-type, same-culture pops

# CONFIGURATION
PRIMARY_CULTURE = None  # auto-pick largest culture per province
PRIMARY_TYPES = [
    "aristocrats", "artisans", "bureaucrats", "capitalists",
    "clergymen", "clerks", "craftsmen", "farmers",
    "labourers", "officers", "slaves", "soldiers"
]
SECONDARY_TYPES = ["farmers"]

ARTISAN_CHUNK_SIZE = 20000
ARTISAN_MIN_CHUNK = 5000   # optional


# PARSER
def parse_pops(block):
    pops = []
    
    pattern = re.compile(
        r'(\w+)\s*=\s*{[^}]*?culture\s*=\s*(\w+)[^}]*?religion\s*=\s*(\w+)[^}]*?size\s*=\s*(\d+)[^}]*?}',
        re.DOTALL
    )

    for match in pattern.finditer(block):
        pop_type, culture, religion, size = match.groups()
        pops.append({
            "type": pop_type,
            "culture": culture,
            "religion": religion,
            "size": int(size)
        })

    return pops


# CORE TRANSFORMATION
def process_province(pops):
    # Step 1: aggregate by (type, culture, religion)
    triple_sum = defaultdict(int)

    for p in pops:
        ptype = normalize_type(p["type"])
        key = (ptype, p["culture"], p["religion"])
        triple_sum[key] += p["size"]

    # Step 2: build religion tracker
    collapsed = defaultdict(int)
    religion_tracker = defaultdict(lambda: defaultdict(int))

    for (ptype, culture, religion), size in triple_sum.items():
        collapsed[(ptype, culture)] += size
        religion_tracker[(ptype, culture)][religion] += size

    final = []

    # Step 3: per (type, culture) decision
    for (ptype, culture), total_size in collapsed.items():

        religions = religion_tracker[(ptype, culture)]

        sorted_rels = sorted(
            religions.items(),
            key=lambda x: x[1],
            reverse=True
        )

        if not sorted_rels:
            continue

        dominant_religion, dominant_size = sorted_rels[0]

        # Shooting Animists
        if dominant_religion == "animist" and len(sorted_rels) > 1:
            second_religion, second_size = sorted_rels[1]

            # ratio check whether animism is absolute majority or not
            if second_size / dominant_size >= 0.8:
                dominant_religion = second_religion

        # Step 4: Redistribute Artisans in predefined chunk size
        if ptype == "artisans":
            chunks = split_artisans(
                total_size,
                ARTISAN_CHUNK_SIZE,
                ARTISAN_MIN_CHUNK
            )

            for chunk_size in chunks:
                final.append({
                    "type": "artisans",
                    "culture": culture,
                    "religion": dominant_religion,
                    "size": chunk_size
                })
        else:
            final.append({
                "type": ptype,
                "culture": culture,
                "religion": dominant_religion,
                "size": total_size
            })

    return final
    
#HELPERS
# Farmer/Labourer Helper
def normalize_type(ptype):
    if ptype in ["farmers", "labourers"]:
        return "farmers"
    return ptype


# Artisan Splitter / Helper
def split_artisans(size, chunk_size, min_chunk):
    chunks = []

    while size > chunk_size:
        chunks.append(chunk_size)
        size -= chunk_size

    # handle remainder
    if size > 0:
        if size < min_chunk and chunks:
            # merge small remainder into last chunk
            chunks[-1] += size
        else:
            chunks.append(size)

    return chunks


# WRITER
def format_province(province_id, pops, indent="\t"):
    lines = [f"{province_id} = {{"]

    for p in pops:
        lines.append(f"{indent}{p['type']} = {{")
        lines.append(f"{indent}\tculture = {p['culture']}")
        lines.append(f"{indent}\treligion = {p['religion']}")
        lines.append(f"{indent}\tsize = {p['size']}")
        lines.append(f"{indent}}}")
        lines.append("")

    lines.append("}")
    return "\n".join(lines)


# MAIN FILE PROCESSOR
def process_file(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Province block regex
    province_pattern = re.compile(
        r'(\d+)\s*=\s*{([^{}]*(?:{[^{}]*}[^{}]*)*)}',
        re.DOTALL
    )

    def replace_province(match):
        province_id = match.group(1)
        original_block = match.group(0)
        inner_content = match.group(2)

        pops = parse_pops(inner_content)

        # do nothing if no pops found
        if not pops:
            return original_block

        new_pops = process_province(pops)

        indent_match = re.search(r'\n(\s*)\w+\s*=', inner_content)
        indent = indent_match.group(1) if indent_match else "\t"

        # Rebuild with collected data
        new_inner = []
        for p in new_pops:
            new_inner.append(f"{indent}{p['type']} = {{")
            new_inner.append(f"{indent}\tculture = {p['culture']}")
            new_inner.append(f"{indent}\treligion = {p['religion']}")
            new_inner.append(f"{indent}\tsize = {p['size']}")
            new_inner.append(f"{indent}}}")
            new_inner.append("")

        new_inner_str = "\n".join(new_inner).rstrip()

        return f"{province_id} = {{\n{new_inner_str}\n}}"

    # Replace province blocks in-place
    new_content = province_pattern.sub(replace_province, content)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(new_content)


# =========================
# RUN
# =========================
if __name__ == "__main__":
    process_file("Madagascar.txt", "Madagascar - simplified.txt")